# Reference Segmentation
Watershed-сегментация частиц с ручной разметкой или SAM2 Hiera Small (32×32).

Для SAM2: **Среда выполнения → Сменить среду выполнения → GPU**. Выполните установку, затем запуск; откройте ссылку Gradio. Для ручной разметки без GPU отключите `use_sam2`. Веса модели скачиваются при первом запросе маркеров.

In [ ]:
#@title Установка
installation_complete = False
use_sam2 = True #@param {type:"boolean"}

import sys
from pathlib import Path

print(f'[1/5] Python {sys.version.split()[0]} — начало установки', flush=True)
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError('Требуется Python 3.10–3.13.')

repo = Path('/content/ReferenceSegmentation')
print('[2/5] Загрузка или обновление приложения', flush=True)
if not repo.exists():
    %shell git clone --depth 1 --branch master https://github.com/Klavikus/ReferenceSegmentation.git /content/ReferenceSegmentation
else:
    origin = %shell git -C /content/ReferenceSegmentation remote get-url origin
    if origin.output.strip() != 'https://github.com/Klavikus/ReferenceSegmentation.git':
        raise RuntimeError('В папке запуска находится другой репозиторий.')
    %shell git -C /content/ReferenceSegmentation pull --ff-only

if use_sam2:
    print('[3/5] Проверка GPU', flush=True)
    %shell "{sys.executable}" -u -c "import torch; print('PyTorch:', torch.__version__); assert torch.cuda.is_available(), 'Выберите GPU в настройках Colab или отключите use_sam2.'; print('GPU:', torch.cuda.get_device_name(0))"
else:
    print('[3/5] Ручная разметка: SAM2 отключён', flush=True)

print('[4/5] Установка зависимостей приложения', flush=True)
%shell "{sys.executable}" -u -m pip install -r /content/ReferenceSegmentation/requirements.txt
if use_sam2:
    print('[5/5] Установка SAM2', flush=True)
    %shell "{sys.executable}" -u -m pip install --no-build-isolation -r /content/ReferenceSegmentation/requirements-sam2.txt
else:
    print('[5/5] Установка SAM2 пропущена', flush=True)

print('Проверка установленных библиотек', flush=True)
%shell "{sys.executable}" -u -c "import cv2, gradio, numpy, pandas; print('NumPy:', numpy.__version__, 'Pandas:', pandas.__version__); print('OpenCV:', cv2.__version__, 'Gradio:', gradio.__version__)"
if use_sam2:
    %shell "{sys.executable}" -u -c "from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator; print('SAM2: импорт успешен')"
installation_complete = True
print('Установка завершена. Выполните следующую ячейку.', flush=True)

In [ ]:
#@title Запуск интерфейса
if not globals().get('installation_complete', False):
    raise RuntimeError('Сначала дождитесь успешного завершения ячейки установки.')

sys.path.insert(0, str(repo))
from backend.application.pipeline_service import PipelineService
from backend.application.registration_service import RegistrationService
from backend.contracts.pipeline_models import PipelineConfig
from backend.infrastructure.markers import Sam2MarkerAdapter
from gradio_gui import GradioGui

sam2_adapter = Sam2MarkerAdapter() if use_sam2 else None
if use_sam2 and not sam2_adapter.is_available():
    raise RuntimeError('SAM2 недоступен. Проверьте вывод установки; после изменения зависимостей перезапустите сессию и выполните обе ячейки снова.')
pipeline = PipelineService(config=PipelineConfig.standard(), sam2_adapter=sam2_adapter)
gui = GradioGui(512, 512)
gui.run(pipeline, share=True, registration_service=RegistrationService())